In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2


# dataset_name = "data_person_1000_zh_fixed_id.json"
dataset_name = "data_person_1000_zh_fixed_gender2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd(), "data", dataset_name), token_model_version="zh_core_web_sm")
print(len(dataset))

def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter


entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "\
f"Min: {min([len(sample.tokens) for sample in dataset])}, "\
f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: " \
f"Min: {min([len(sample.full_text) for sample in dataset])}, "\
f"Max: {max([len(sample.full_text) for sample in dataset])}")

/home/capcom/anaconda3/envs/presidio-research/lib/python3.10/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/capcom/anaconda3/envs/presidio-research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


stanza and spacy_stanza are not installed
Flair is not installed by default


tokenizing input:   0%|          | 0/2 [00:00<?, ?it/s]

loading model zh_core_web_sm


tokenizing input: 100%|██████████| 2/2 [00:00<00:00,  3.38it/s]

2
Count per entity:
[('O', 143), ('STREET_ADDRESS', 11), ('PERSON', 6), ('EMAIL_ADDRESS', 6),
 ('AGE', 3), ('GENDER', 2), ('ID', 2), ('PHONE_NUMBER', 2)]

Min and max number of tokens in dataset: Min: 82, Max: 93
Min and max sentence length in dataset: Min: 185, Max: 206


In [3]:
from gliner_engine import GLiNERNlpEngine, GLiNERRecognizer, GLINER_LABEL_MAPPING
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry


# MODEL_NAME = "urchade/gliner_multi-v2.1"
MODEL_NAME = "/home/capcom/models/gliner-multi-edu"
THRESHOLD = 0.5
LABELS = [
    "person",
    "location",
    "age",
    "email address",
    "phone number",
    "personal identification number",
    "credit_card",
    "gender"
]

nlp_engine = GLiNERNlpEngine(
    model_name=MODEL_NAME,
    labels=LABELS,
    label_mapping=GLINER_LABEL_MAPPING,
    threshold=THRESHOLD,
)
recognizer = GLiNERRecognizer(nlp_engine=nlp_engine, score_threshold=THRESHOLD)

registry = RecognizerRegistry(supported_languages=["zh"])
registry.add_recognizer(recognizer)


from presidio_analyzer import PatternRecognizer, Pattern

email_pattern = Pattern(
    name="email",
    regex=r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    score=0.85,
)


email_recognizer = PatternRecognizer(
    supported_entity="EMAIL_ADDRESS",
    patterns=[email_pattern],
    name="EmailRecognizer",
    supported_language="zh",
)

registry.add_recognizer(email_recognizer)

phone_pattern = Pattern(
    name="cn_phone",
    regex=r"(?<!\d)(?:\+?86[- ]?)?1[3-9]\d{9}(?!\d)",
    score=0.9,
)

phone_recognizer = PatternRecognizer(
    supported_entity="PHONE_NUMBER",
    patterns=[phone_pattern],
    name="PhoneRecognizer",
    supported_language="zh",
)

registry.add_recognizer(phone_recognizer)

id_pattern = Pattern(
    name="cn_id",
    regex=r"(?<!\d)\d{17}[\dXx](?!\d)",
    score=0.95,
)

id_recognizer = PatternRecognizer(
    supported_entity="ID",
    patterns=[id_pattern],
    name="IdRecognizer",
    supported_language="zh",
)

registry.add_recognizer(id_recognizer)

gender_pattern = Pattern(
    name="cn_gender",
    regex=(
        r"(男性|女性|先生|女士|小姐)"
        r"|(?:(?<=性别)|(?<=性别为)|(?<=性别是)|(?<=\d岁)|(?<=\d岁的))(男|女)"
    ),
    score=0.8,
)

gender_recognizer = PatternRecognizer(
    supported_entity="GENDER",
    patterns=[gender_pattern],
    name="GenderRecognizer",
    supported_language="zh",
)

registry.add_recognizer(gender_recognizer)

analyzer_engine = AnalyzerEngine(
    nlp_engine=nlp_engine,
    registry=registry,
    supported_languages=["zh"],
)

pprint(analyzer_engine.get_supported_entities("zh"), compact=True)
pprint([rec.name for rec in analyzer_engine.registry.get_recognizers("zh", all_fields=True)], compact=True)

Loading GLiNER model from /home/capcom/models/gliner-multi-edu on CPU...
GLiNER model loaded in 4.1s
['CREDIT_CARD', 'LOCATION', 'PERSON', 'ID', 'PHONE_NUMBER', 'ORGANIZATION',
 'EMAIL_ADDRESS', 'AGE', 'GENDER']
['GLiNERRecognizer', 'EmailRecognizer', 'PhoneRecognizer', 'IdRecognizer',
 'GenderRecognizer']


In [4]:
from presidio_anonymizer import AnonymizerEngine

anonymizer = AnonymizerEngine()

cn_text = "沈雨桐，46岁女性牙科医师，近期出现频繁口渴、尿量增多和视力模糊等症状后被诊断为糖尿病。现居住于浙江省杭州市西湖区文三路81号，可通过邮箱shen.yutong@163.com或手机13857123456联系。身份证号码为330106197702283456。诊断由张雪梅医生作出，目前用药包括青霉素。财务方面，沈雨桐年收入为1,327,054.2元，信用评分为687分。近期交易记录包含一笔中国银行内部资金转账。"

print(cn_text)
results_chinese = analyzer_engine.analyze(text=cn_text, language="zh")
print("results_chinese:")
print(results_chinese)

anonymized_cn_text = anonymizer.anonymize(text=cn_text, analyzer_results=results_chinese)
print("anonymized_cn_text:")
print(anonymized_cn_text)

沈雨桐，46岁女性牙科医师，近期出现频繁口渴、尿量增多和视力模糊等症状后被诊断为糖尿病。现居住于浙江省杭州市西湖区文三路81号，可通过邮箱shen.yutong@163.com或手机13857123456联系。身份证号码为330106197702283456。诊断由张雪梅医生作出，目前用药包括青霉素。财务方面，沈雨桐年收入为1,327,054.2元，信用评分为687分。近期交易记录包含一笔中国银行内部资金转账。
Running GLiNER inference...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


GLiNER inference finished in 0.8s
results_chinese:
[type: ID, start: 111, end: 129, score: 0.95, type: PERSON, start: 0, end: 3, score: 0.9499878287315369, type: PHONE_NUMBER, start: 91, end: 102, score: 0.9350223541259766, type: PERSON, start: 156, end: 159, score: 0.9064028859138489, type: EMAIL_ADDRESS, start: 69, end: 88, score: 0.85, type: GENDER, start: 7, end: 9, score: 0.8, type: AGE, start: 4, end: 7, score: 0.7179549932479858, type: PERSON, start: 133, end: 136, score: 0.6939047574996948]
anonymized_cn_text:
text: <PERSON>，<AGE><GENDER>牙科医师，近期出现频繁口渴、尿量增多和视力模糊等症状后被诊断为糖尿病。现居住于浙江省杭州市西湖区文三路81号，可通过邮箱<EMAIL_ADDRESS>或手机<PHONE_NUMBER>联系。身份证号码为<ID>。诊断由<PERSON>医生作出，目前用药包括青霉素。财务方面，<PERSON>年收入为1,327,054.2元，信用评分为687分。近期交易记录包含一笔中国银行内部资金转账。
items:
[
    {'start': 159, 'end': 167, 'entity_type': 'PERSON', 'text': '<PERSON>', 'operator': 'replace'},
    {'start': 131, 'end': 139, 'entity_type': 'PERSON', 'text': '<PERSON>', 'operator': 'replace'},
    {'start': 123, 'end': 127, 'entity_type':

In [6]:
from presidio_evaluator.models import PresidioAnalyzerWrapper

entities_mapping = PresidioAnalyzerWrapper.presidio_entities_map
entities_mapping['GENDER'] = 'GENDER'
pprint(entities_mapping, compact=True)

aligned_dataset = SpanEvaluator.align_entity_types(
    dataset,
    entities_mapping=entities_mapping,
    allow_missing_mappings=True,
)
new_entity_counts = get_entity_counts(aligned_dataset)
pprint(new_entity_counts.most_common(), compact=True)

{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GENDER': 'GENDER',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'PREFIX': 'TITLE',
 'SSN': 'U

In [7]:
# Set up the experiment tracker to log the experiment for reproducibility
experiment = get_experiment_tracker()


# Create the evaluator object
evaluator = SpanEvaluator(model=analyzer_engine, iou_threshold=0.7)


# Track model and dataset params
params = {"dataset_name": dataset_name, "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

## Run experiment

evaluation_results = evaluator.evaluate_all(dataset, language="zh")
results = evaluator.calculate_score(evaluation_results)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, 
                                labels=entities)

# end experiment
experiment.end()

# Note that the experiment params and metrics are saved locally

--------
Entities supported by this Presidio Analyzer instance:
CREDIT_CARD, LOCATION, PERSON, ID, PHONE_NUMBER, ORGANIZATION, EMAIL_ADDRESS, AGE, GENDER
Running model PresidioAnalyzerWrapper on dataset...
Running GLiNER inference...


/home/capcom/presidio/presidio-research/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


GLiNER inference finished in 0.7s
Running GLiNER inference...
GLiNER inference finished in 0.8s
Finished running model on dataset
saving experiment data to /home/capcom/presidio/Qwen3Guard/experiment_20260320-203045.json


In [9]:
plotter = Plotter(
    results=results,
    model_name=evaluator.model.name,
    save_as="svg",
    beta=2,
)
output_folder = Path(Path.cwd(), "plotter_output_gliner")
plotter.plot_scores(output_folder=output_folder)
output_folder

PosixPath('/home/capcom/presidio/Qwen3Guard/plotter_output_gliner')

In [14]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity=["LOCATION"])
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

No errors of type ErrorType.FP and entity ['LOCATION'] were found


TypeError: 'NoneType' object is not subscriptable

In [ ]:
fps_df = ModelError.get_fps_dataframe(results.model_errors)

In [19]:
fps_df['prediction'].unique()

array(['ORGANIZATION', 'PERSON', 'AGE'], dtype=object)